# Time-Use Profiles: Archetype Classifier Model

Run the multiday archetype-classifier workflow and optionally export hub-ready artifacts.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import sys

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name.lower() == "notebooks" else CWD
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.timeuse_profiles import (
    build_hub_timeuse_package,
    export_hub_timeuse_package,
    prepare_timeuse_data,
    run_multiday_classifier_workflow,
)


In [10]:
LIMIT_TO_SELECTED_DAS = True
N_SELECTED_DAS = 10
OUTPUT_TAG = "archetype_classifier_da50" if LIMIT_TO_SELECTED_DAS else "archetype_classifier"
EXPORT_HUB = True
OUT_DIR = PROJECT_ROOT / "data" / "processed" / "timeuse_profiles"
HUB_OUT_DIR = OUT_DIR / "hub"


In [23]:
from src.timeuse_profiles import TimeUseConfig, prepare_timeuse_data, run_multiday_classifier_workflow

config = TimeUseConfig(
    n_archetypes=2,
    base_feature_cols=(
        "sex_std",
        "age_band",
        "hhsize_cat",
        "has_children",
        "economic_activity_cat",
        "household_comp_proxy_cat",
    ),
    exact_match_cols=(
        "sex_std",
        "age_band",
        "hhsize_cat",
        "has_children",
    ),
    fallback_match_sets=(
        ("sex_std", "age_band", "has_children", "economic_activity_cat"),
        ("sex_std", "age_band", "economic_activity_cat", "household_comp_proxy_cat"),
        ("sex_std", "age_band", "household_comp_proxy_cat"),
        ("age_band", "economic_activity_cat"),
        ("age_band",),
    ),
)


In [24]:

prepared = prepare_timeuse_data(
    config=config,
    limit_to_selected_das=LIMIT_TO_SELECTED_DAS,
    n_selected_das=N_SELECTED_DAS,
)

workflow = run_multiday_classifier_workflow(prepared)


In [25]:

workflow["all_metrics"].to_csv(OUT_DIR / f"archetype_classifier_metrics_{OUTPUT_TAG}_all_daytypes.csv", index=False)
workflow["all_district_profiles"].to_csv(OUT_DIR / f"district_hourly_profiles_{OUTPUT_TAG}_all_daytypes.csv", index=False)
workflow["all_district_archetypes"].to_csv(OUT_DIR / f"district_archetype_expected_counts_{OUTPUT_TAG}_all_daytypes.csv", index=False)
workflow["all_centroids"].to_csv(OUT_DIR / f"tus_archetype_centroids_{OUTPUT_TAG}_all_daytypes.csv", index=False)
if not workflow["stability"].empty:
    workflow["stability"].to_csv(OUT_DIR / f"district_hourly_weekday_weekend_stability_{OUTPUT_TAG}.csv", index=False)
if not workflow["stability_summary"].empty:
    workflow["stability_summary"].to_csv(OUT_DIR / f"district_hourly_weekday_weekend_stability_summary_{OUTPUT_TAG}.csv", index=False)

display(workflow["all_metrics"])
display(workflow["stability_summary"])


,n_train,n_test,n_classes,accuracy,balanced_accuracy,log_loss,day_type,n_synthetic_people,n_tus_respondents,n_archetypes
0,6670,2224,2,0.685982,0.720126,0.609273,weekday,5102,8894,2
1,2581,861,2,0.598783,0.568589,0.660570,weekend,5102,3442,2


,metric,mean_abs_delta,p95_abs_delta
0,share_home_awake,0.034285,0.086059
1,share_sleep,0.024034,0.112300
2,share_away,0.042930,0.132918
3,share_home_total,0.042930,0.132918


In [4]:
if EXPORT_HUB:
    for day_type, result in workflow["results_by_day"].items():
        package = build_hub_timeuse_package(
            syn_people=result["syn_people"],
            district_profiles=result["district_profiles"].drop(columns=["day_type"], errors="ignore"),
            centroid_profiles=result["centroid_profiles"].drop(columns=["day_type"], errors="ignore"),
            district_archetype_counts=result["district_archetype_long"],
            method="archetype_classifier",
            day_type=day_type,
            output_tag=f"{OUTPUT_TAG}_{day_type}",
        )
        outputs = export_hub_timeuse_package(
            package,
            out_dir=HUB_OUT_DIR,
            stem=f"{OUTPUT_TAG}_{day_type}",
        )
        print(day_type, outputs)
else:
    print("Set EXPORT_HUB = True to write hub-ready artifacts.")


weekday {'da_sociodemographic_profile': WindowsPath('C:/Users/m_hamsai/OneDrive - Concordia University - Canada/PhD Research/Codes and Projects/DSM and SD/data/processed/timeuse_profiles/hub/archetype_classifier_weekday__da_sociodemographic_profile.parquet'), 'synthetic_population': WindowsPath('C:/Users/m_hamsai/OneDrive - Concordia University - Canada/PhD Research/Codes and Projects/DSM and SD/data/processed/timeuse_profiles/hub/archetype_classifier_weekday__synthetic_population.parquet'), 'area_behavioral_archetype': WindowsPath('C:/Users/m_hamsai/OneDrive - Concordia University - Canada/PhD Research/Codes and Projects/DSM and SD/data/processed/timeuse_profiles/hub/archetype_classifier_weekday__area_behavioral_archetype.parquet'), 'tus_activity_profile': WindowsPath('C:/Users/m_hamsai/OneDrive - Concordia University - Canada/PhD Research/Codes and Projects/DSM and SD/data/processed/timeuse_profiles/hub/archetype_classifier_weekday__tus_activity_profile.parquet'), 'activity_schedule_

In [26]:
import numpy as np
import pandas as pd
import plotly.express as px

# Expected in memory from notebook 07:
# - workflow
# - OUTPUT_TAG
# - OUT_DIR
# - HUB_OUT_DIR (optional)

results_by_day = workflow["results_by_day"]
all_metrics = workflow["all_metrics"].copy()
all_district_profiles = workflow["all_district_profiles"].copy()
all_district_archetypes = workflow["all_district_archetypes"].copy()
all_centroids = workflow["all_centroids"].copy()
stability = workflow["stability"].copy()
stability_summary = workflow["stability_summary"].copy()

PROFILE_COLS = ["share_home_awake", "share_sleep", "share_away", "share_home_total"]

print("Available day types:", list(results_by_day.keys()))
display(all_metrics)

# 1) Classifier performance by day type
if not all_metrics.empty:
    metric_cols = [c for c in ["accuracy", "balanced_accuracy", "log_loss"] if c in all_metrics.columns]
    metrics_long = all_metrics.melt(
        id_vars=[c for c in ["day_type"] if c in all_metrics.columns],
        value_vars=metric_cols,
        var_name="metric",
        value_name="value",
    )
    fig_metrics = px.bar(
        metrics_long,
        x="day_type",
        y="value",
        color="metric",
        barmode="group",
        title=f"Classifier performance by day type ({OUTPUT_TAG})",
    )
    fig_metrics.show()

# 2) Mean district profiles by day type
if not all_district_profiles.empty:
    mean_profiles = (
        all_district_profiles
        .groupby(["day_type", "hour"], as_index=False)[PROFILE_COLS]
        .mean()
    )
    plot_profiles = mean_profiles.melt(
        id_vars=["day_type", "hour"],
        value_vars=PROFILE_COLS,
        var_name="state",
        value_name="share",
    )
    fig_profiles = px.line(
        plot_profiles,
        x="hour",
        y="share",
        color="state",
        line_dash="day_type",
        title=f"Mean district profiles from archetype classifier ({OUTPUT_TAG})",
    )
    fig_profiles.show()

    display(mean_profiles.head())

# 3) Archetype centroid profiles
if not all_centroids.empty:
    centroid_plot = all_centroids.melt(
        id_vars=["day_type", "archetype_id", "hour"],
        value_vars=["share_home_awake", "share_sleep", "share_away"],
        var_name="state",
        value_name="share",
    )
    fig_centroids = px.line(
        centroid_plot,
        x="hour",
        y="share",
        color="state",
        line_dash="archetype_id",
        facet_row="day_type",
        title=f"TUS schedule archetype centroids ({OUTPUT_TAG})",
        height=900,
    )
    fig_centroids.show()

# 4) District archetype expected counts heatmap
if not all_district_archetypes.empty:
    for day_type in sorted(all_district_archetypes["day_type"].astype(str).unique()):
        sub = all_district_archetypes.loc[all_district_archetypes["day_type"].astype(str) == day_type].copy()
        archetype_cols = [c for c in sub.columns if c not in {"district_id", "day_type"}]
        if archetype_cols:
            heat_df = sub.set_index("district_id")[archetype_cols].apply(pd.to_numeric, errors="coerce")
            if not heat_df.empty:
                fig_arch = px.imshow(
                    heat_df,
                    aspect="auto",
                    color_continuous_scale="YlGnBu",
                    title=f"District archetype expected counts heatmap: {day_type} ({OUTPUT_TAG})",
                )
                fig_arch.show()

# 5) Top districts by mean home share
if not all_district_profiles.empty:
    district_rank = (
        all_district_profiles
        .groupby(["day_type", "district_id"], as_index=False)["share_home_total"]
        .mean()
        .rename(columns={"share_home_total": "mean_home_total"})
    )
    fig_rank = px.bar(
        district_rank.sort_values("mean_home_total", ascending=False).head(30),
        x="district_id",
        y="mean_home_total",
        color="day_type",
        barmode="group",
        title=f"Top districts by mean home share ({OUTPUT_TAG})",
    )
    fig_rank.show()

# 6) Weekday/weekend stability summary
if not stability_summary.empty:
    fig_stability_bar = px.bar(
        stability_summary,
        x="metric",
        y="mean_abs_delta",
        title=f"Weekday vs weekend stability summary ({OUTPUT_TAG})",
    )
    fig_stability_bar.show()
    display(stability_summary)

# 7) Weekday/weekend stability by hour
if not stability.empty:
    hourly_stability_cols = [c for c in stability.columns if c.endswith("_abs_delta")]
    if hourly_stability_cols and "hour" in stability.columns:
        stability_hourly = stability.groupby("hour", as_index=False)[hourly_stability_cols].mean()
        stability_hourly = stability_hourly.melt(
            id_vars="hour",
            var_name="metric",
            value_name="mean_abs_delta",
        )
        fig_stability_line = px.line(
            stability_hourly,
            x="hour",
            y="mean_abs_delta",
            color="metric",
            title=f"Weekday vs weekend stability by hour ({OUTPUT_TAG})",
        )
        fig_stability_line.show()

# 8) Optional: inspect hub export manifest(s)
if "HUB_OUT_DIR" in globals() and HUB_OUT_DIR.exists():
    manifest_files = sorted(HUB_OUT_DIR.glob(f"{OUTPUT_TAG}*__manifest.json"))
    print("Hub manifests found:", len(manifest_files))
    for mf in manifest_files[:5]:
        print(mf.name)


Available day types: ['weekday', 'weekend']


,n_train,n_test,n_classes,accuracy,balanced_accuracy,log_loss,day_type,n_synthetic_people,n_tus_respondents,n_archetypes
0,6670,2224,2,0.685982,0.720126,0.609273,weekday,5102,8894,2
1,2581,861,2,0.598783,0.568589,0.660570,weekend,5102,3442,2


,day_type,hour,share_home_awake,share_sleep,share_away,share_home_total
0,weekday,0,0.117395,0.857058,0.025546,0.974454
1,weekday,1,0.076737,0.904191,0.019072,0.980928
2,weekday,2,0.060584,0.922490,0.016925,0.983075
3,weekday,3,0.056643,0.926665,0.016692,0.983308
4,weekday,4,0.093291,0.879550,0.027158,0.972842


,metric,mean_abs_delta,p95_abs_delta
0,share_home_awake,0.034285,0.086059
1,share_sleep,0.024034,0.112300
2,share_away,0.042930,0.132918
3,share_home_total,0.042930,0.132918


Hub manifests found: 0


In [20]:
import json
import numpy as np
import pandas as pd
import plotly.express as px

from src.synthpop import build_da_gdf, norm_code

DAY_TYPE_TO_MAP = "weekday"   # weekday or weekend
MAP_METRIC = "mean_home_total"  # mean_home_total, max_home_total, mean_away
TOP_N_LABELS = 20

# 1) Build DA geometry
da_gdf, da_col = build_da_gdf(project_root=PROJECT_ROOT)
da_gdf = da_gdf.copy()
da_gdf["district_id"] = da_gdf[da_col].map(norm_code).astype(str)

# 2) Build district summary from classifier outputs
district_summary = (
    all_district_profiles
    .loc[all_district_profiles["day_type"].astype(str) == DAY_TYPE_TO_MAP]
    .groupby("district_id", as_index=False)
    .agg(
        mean_home_total=("share_home_total", "mean"),
        max_home_total=("share_home_total", "max"),
        mean_away=("share_away", "mean"),
    )
)
district_summary["district_id"] = district_summary["district_id"].astype(str)

# 3) Join geometry to classifier outputs
map_gdf = da_gdf.merge(district_summary, on="district_id", how="inner")
print("Matched geometry rows:", len(map_gdf))
display(map_gdf[[da_col, "district_id", MAP_METRIC]].head())

if map_gdf.empty:
    raise ValueError(
        "No DA geometry matched the classifier district_id values. "
        "Check that district_id in notebook 7 comes from the same DA coding system as the synthpop geometry."
    )

# 4) Reproject if needed
if map_gdf.crs is not None and str(map_gdf.crs).lower() not in {"epsg:4326", "4326"}:
    map_gdf = map_gdf.to_crs(epsg=4326)

map_gdf = map_gdf.reset_index(drop=True)
map_gdf["plot_id"] = map_gdf.index.astype(str)
geojson = json.loads(map_gdf.to_json())

center = {
    "lat": float(map_gdf.geometry.centroid.y.mean()),
    "lon": float(map_gdf.geometry.centroid.x.mean()),
}

# 5) Choropleth map
fig_map = px.choropleth_mapbox(
    map_gdf,
    geojson=geojson,
    locations="plot_id",
    featureidkey="properties.plot_id",
    color=MAP_METRIC,
    hover_name="district_id",
    hover_data={
        "mean_home_total": ":.3f",
        "max_home_total": ":.3f",
        "mean_away": ":.3f",
        "plot_id": False,
    },
    color_continuous_scale="YlGnBu",
    mapbox_style="carto-positron",
    center=center,
    zoom=9,
    opacity=0.7,
    title=f"{DAY_TYPE_TO_MAP.title()} DA map of {MAP_METRIC} ({OUTPUT_TAG})",
    height=750,
)
fig_map.update_traces(marker_line_width=0.15, marker_line_color="white")
fig_map.show()

# 6) Top districts overlay
top_df = map_gdf.sort_values(MAP_METRIC, ascending=False).head(TOP_N_LABELS).copy()
top_df["lon"] = top_df.geometry.centroid.x
top_df["lat"] = top_df.geometry.centroid.y

fig_top = px.scatter_mapbox(
    top_df,
    lat="lat",
    lon="lon",
    text="district_id",
    size=MAP_METRIC,
    color=MAP_METRIC,
    color_continuous_scale="Turbo",
    mapbox_style="carto-positron",
    center=center,
    zoom=9,
    title=f"Top {TOP_N_LABELS} DAs by {MAP_METRIC} ({DAY_TYPE_TO_MAP})",
    height=750,
)
fig_top.update_traces(textposition="top right")
fig_top.show()


Matched geometry rows: 10


,DAUID,district_id,mean_home_total
0,24660220,24660220,0.733439
1,24660452,24660452,0.729798
2,24661086,24661086,0.699447
3,24661729,24661729,0.756984
4,24661818,24661818,0.713795


C:\Users\m_hamsai\AppData\Local\Temp\ipykernel_18816\3585052613.py:50: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.


C:\Users\m_hamsai\AppData\Local\Temp\ipykernel_18816\3585052613.py:51: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.


C:\Users\m_hamsai\AppData\Local\Temp\ipykernel_18816\3585052613.py:55: DeprecationWarning:

*choropleth_mapbox* is deprecated! Use *choropleth_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



C:\Users\m_hamsai\AppData\Local\Temp\ipykernel_18816\3585052613.py:81: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.


C:\Users\m_hamsai\AppData\Local\Temp\ipykernel_18816\3585052613.py:82: UserWarning:

Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.


C:\Users\m_hamsai\AppData\Local\Temp\ipykernel_18816\3585052613.py:84: DeprecationWarning:

*scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/



In [27]:
import pandas as pd
import plotly.express as px

DAY_TYPE_TO_SHOW = "weekday"

centroid_profiles = workflow["results_by_day"][DAY_TYPE_TO_SHOW]["centroid_profiles"].copy()

if "share_home_total" not in centroid_profiles.columns:
    centroid_profiles["share_home_total"] = (
        pd.to_numeric(centroid_profiles["share_home_awake"], errors="coerce").fillna(0)
        + pd.to_numeric(centroid_profiles["share_sleep"], errors="coerce").fillna(0)
    )

for archetype_id in sorted(pd.to_numeric(centroid_profiles["archetype_id"], errors="coerce").dropna().unique()):
    sub = centroid_profiles.loc[centroid_profiles["archetype_id"] == archetype_id].copy()

    plot_df = sub.melt(
        id_vars=["archetype_id", "hour"],
        value_vars=["share_home_awake", "share_sleep", "share_away", "share_home_total"],
        var_name="state",
        value_name="share",
    )

    fig = px.line(
        plot_df,
        x="hour",
        y="share",
        color="state",
        title=f"Archetype {int(archetype_id)} profile: {DAY_TYPE_TO_SHOW} ({OUTPUT_TAG})",
        height=450,
    )
    fig.update_traces(line=dict(width=4))
    fig.show()
